# ScaleUp — Piano Transcription Server

**לפני שמתחילים:** לך ל-`Runtime → Change runtime type` ובחר **GPU (T4)**.

הרץ את 3 התאים לפי הסדר:
- **תא 1** — התקנת חבילות (~2 דק')
- **תא 2** — טעינת מודל (~3 דק')
- **תא 3** — הפעלת שרת → תקבל URL

העתק את ה-URL ל-ScaleUp → Audio to Tab → ⚙️ Advanced → MT3 Server URL

In [ ]:
#@title תא 1 — התקנת חבילות { display-mode: "form" }
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'piano-transcription-inference',
    'flask',
    'librosa',
], check=True)

print('✅ חבילות הותקנו')

In [ ]:
#@title תא 2 — טעינת המודל (~3 דקות) { display-mode: "form" }
import torch, tempfile, os, numpy as np
from piano_transcription_inference import PianoTranscription, sample_rate as PT_SR, load_audio

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('טוען מודל... (מוריד ~130 MB בפעם הראשונה)')

TRANSCRIPTOR = PianoTranscription(device=DEVICE, checkpoint_path=None)

def transcribe_audio(audio_path: str):
    audio, _ = load_audio(audio_path, sr=PT_SR, mono=True)
    with tempfile.NamedTemporaryFile(suffix='.mid', delete=False) as tmp:
        result = TRANSCRIPTOR.transcribe(audio, tmp.name)
        os.unlink(tmp.name)
    return result['est_note_events']

def note_sequence_to_json(events) -> list:
    results = []
    for onset, offset, pitch, velocity in events:
        freq = 440.0 * (2 ** ((int(pitch) - 69) / 12.0))
        results.append({
            'startTime':  round(float(onset), 4),
            'endTime':    round(float(offset), 4),
            'midiNote':   int(pitch),
            'confidence': round(float(velocity) / 127.0, 3),
            'frequency':  round(freq, 3),
        })
    results.sort(key=lambda n: n['startTime'])
    return results

print('✅ מודל נטען!')

In [ ]:
#@title תא 3 — הפעלת שרת + URL ציבורי { display-mode: "form" }
#
# השרת רץ כל עוד התא פועל.
# לעצירה: Runtime → Interrupt execution

import os, tempfile, threading, subprocess, re, time
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.after_request
def cors(r):
    r.headers['Access-Control-Allow-Origin']  = '*'
    r.headers['Access-Control-Allow-Methods'] = 'POST,GET,OPTIONS'
    r.headers['Access-Control-Allow-Headers'] = '*'
    return r

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'model': 'piano_transcription'})

@app.route('/transcribe', methods=['POST', 'OPTIONS'])
def transcribe():
    if request.method == 'OPTIONS':
        return '', 204
    if 'audio' not in request.files:
        return jsonify({'error': 'missing audio field'}), 400
    f   = request.files['audio']
    ext = os.path.splitext(f.filename or 'audio.wav')[-1] or '.wav'
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
        f.save(tmp.name)
        path = tmp.name
    try:
        events = transcribe_audio(path)
        notes  = note_sequence_to_json(events)
        return jsonify({'notes': notes, 'count': len(notes)})
    except Exception as e:
        return jsonify({'error': str(e)}), 500
    finally:
        os.unlink(path)

os.system('fuser -k 5000/tcp 2>/dev/null || true')
time.sleep(1)

threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False),
    daemon=True
).start()
time.sleep(2)
print('Flask פועל על פורט 5000')

def _start_tunnel():
    proc = subprocess.Popen(
        ['ssh', '-o', 'StrictHostKeyChecking=no',
         '-o', 'ServerAliveInterval=30',
         '-R', '80:localhost:5000',
         'nokey@localhost.run'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in proc.stdout:
        txt = line.decode(errors='ignore')
        m = re.search(r'https://[a-z0-9\\-]+\\.lhr\\.life', txt)
        if m:
            url = m.group()
            print('\n' + '='*55)
            print('✅  השרת פעיל!')
            print(f'   URL: {url}')
            print('\nהדבק ב-ScaleUp → Audio to Tab → ⚙️ Advanced:')
            print(f'  {url}')
            print('='*55)
            return

threading.Thread(target=_start_tunnel, daemon=True).start()
print('מחכה ל-URL... (~10 שניות)')